In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
pip install seaborn


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.1 MB 11.8 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.1 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.1 MB 13.2 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 11.9 MB/s eta 0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 30.9 MB/s eta 0:00:00


In [2]:
pip install numpy==1.24.4


   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   ------------- -------------------------- 5.0/14.8 MB 25.2 MB/s eta 0:00:01
   -------------------------- ------------- 10.0/14.8 MB 24.9 MB/s eta 0:00:01
   ---------------------------------------  14.7/14.8 MB 24.3 MB/s eta 0:00:01
   ---------------------------------------- 14.8/14.8 MB 22.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.


In [4]:
IMG_SIZE = 48

def apply_thresholding(image):
    image = (image * 255).astype(np.uint8)

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) 

    _, thresholded = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    thresholded_rgb = np.stack([thresholded] * 3, axis=-1)

    return thresholded_rgb

def load_images_from_directory(directory):
    images = []
    labels = []
    label_map = {'ADHD-Hyperactive': 0, 'Typically Developing Children': 1} 
    
    for class_name in os.listdir(directory):
        class_folder = os.path.join(directory, class_name)
        if os.path.isdir(class_folder):
            for filename in os.listdir(class_folder):
                if filename.endswith(".jpg") or filename.endswith(".png"):
                    img = load_img(os.path.join(class_folder, filename), target_size=(IMG_SIZE, IMG_SIZE))
                    img = img_to_array(img) / 255.0 
                    img = apply_thresholding(img)  
                    
                    images.append(img)
                    labels.append(label_map[class_name])
    
    return np.array(images), np.array(labels)

train_dir = r"C:\Users\pooja\Desktop\ADHD Preprocessed Data\train"
val_dir = r"C:\Users\pooja\Desktop\ADHD Preprocessed Data\val"
test_dir = r"C:\Users\pooja\Desktop\ADHD Preprocessed Data\test"

xtrain, ytrain = load_images_from_directory(train_dir)
xval, yval = load_images_from_directory(val_dir)
xtest, ytest = load_images_from_directory(test_dir)

In [5]:
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE,3))
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid') 
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

80134624/80134624 [==============================] - 6s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg19 (Functional)          (None, 1, 1, 512)         20024384  
                                                                 
 global_average_pooling2d (G  (None, 512)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               65664     
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 20,090,177
Trainable params: 65,793
Non-trainable params: 20,024,384
_________________________________________________________________


In [6]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(xtrain, ytrain, 
          batch_size=32, 
          epochs=10, 
          validation_data=(xval, yval),
          callbacks=[early_stopping])

Epoch 1/10
1563/1563 [==============================] - 261s 166ms/step - loss: 0.8686 - accuracy: 0.7264 - val_loss: 0.5566 - val_accuracy: 0.7484
Epoch 2/10
1563/1563 [==============================] - 511s 327ms/step - loss: 0.4123 - accuracy: 0.8084 - val_loss: 0.4957 - val_accuracy: 0.7638
Epoch 3/10
1563/1563 [==============================] - 243s 156ms/step - loss: 0.3477 - accuracy: 0.8430 - val_loss: 0.4631 - val_accuracy: 0.7784
Epoch 4/10
1563/1563 [==============================] - 334s 214ms/step - loss: 0.3089 - accuracy: 0.8635 - val_loss: 0.5056 - val_accuracy: 0.7938
Epoch 5/10
1563/1563 [==============================] - 256s 164ms/step - loss: 0.2713 - accuracy: 0.8824 - val_loss: 0.4915 - val_accuracy: 0.8052
Epoch 6/10
1563/1563 [==============================] - 238s 152ms/step - loss: 0.2438 - accuracy: 0.8974 - val_loss: 0.5137 - val_accuracy: 0.7946
